In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow

## First, we load in some data.

Katherine Qi of the UW SeaFlow group has provided us with some **flow cytometer data** for clustering. The data are abundance and optical properties of phytoplankton and there are several attributes for each particle, such as normalized scatter, red, orange, and green. These are measurements from the instrument itself from light scattering. Other parameters, like diam (diameter) and Qc (carbon quota) are estimated from the light scatter measurements.

In [ ]:
# Download the data and then load it in

wget.download("https://www.dropbox.com/s/dwa82x6xhjkhyw8/ug3_FCM_distribution.feather?dl=1")
underway_g3 = pd.read_feather("ug3_FCM_distribution.feather")
underway_g3.head()

*Each file is a single sample taken at a certain time, location, and depth. There are also replicates, or even triplicates, run on the same spatiotemporal scale to get uncertainty estimations on the instrument. We can either ignore the replicates/triplicates or take the mean.*

In [ ]:
files = list(pd.unique(underway_g3['filename']))
print(files)

## Let's examine the dataset

In [ ]:
singleSample = underway_g3[underway_g3['filename']==files[0]]

In [ ]:
df = singleSample[['norm.scatter','norm.red','norm.orange','norm.green','depth']].copy()
df['norm.scatter'] = np.log10(df['norm.scatter'])
df['norm.red'] = np.log10(df['norm.red'])
df['norm.orange'] = np.log10(df['norm.orange'])

# Pairplot with matplotlib
columns = df.columns.tolist()
n = len(columns)

fig, axes = plt.subplots(n, n, figsize=(10, 10))

for i, col_y in enumerate(columns):
    for j, col_x in enumerate(columns):
        ax = axes[i, j]
        if i == j:
            ax.hist(df[col_x], bins=30, color='steelblue', edgecolor='k', alpha=0.7)
        else:
            ax.scatter(df[col_x], df[col_y], c=df['norm.green'], cmap='viridis', s=5, alpha=0.5)
        if i == n - 1:
            ax.set_xlabel(col_x)
        if j == 0:
            ax.set_ylabel(col_y)

plt.tight_layout()
plt.show()


In [ ]:
n = len(df)
p = 3 #()
print('We have {:d} data points, and each one has {:d} features'.format(n, p))

In [ ]:
# 3D scatter plot with matplotlib
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(df['norm.scatter'], 
           df['norm.red'], 
           df['norm.orange'],
           s=5, alpha=0.6)

ax.set_xlabel('norm.scatter')
ax.set_ylabel('norm.red')
ax.set_zlabel('norm.orange')
ax.set_box_aspect(None, zoom=0.85)
plt.show()

We are going to write a function that randomly creates *k* centers.

In [ ]:
data = np.zeros(shape=(n,p))
data[:,0] = df['norm.scatter']
data[:,1] = df['norm.red']
data[:,2] = df['norm.orange']

In [ ]:
def init_centers(data, k):
    """
    """
    # Initialize centroids
    centers = np.zeros((k, np.shape(data)[1]))
    # Loop on k centers
    for i in range(0, k):
        # Generate p random values between 0 and 1
        dist = np.random.uniform(size=np.shape(data)[1])
        # Use the random values to generate a point within the range of values taken by the data
        centers[i, :] = np.min(data, axis=0) + (np.max(data, axis=0) - np.min(data, axis=0)) * dist
    return centers

To be able to assign each data point to the closest centroid, we need to define the distance between two data points. 

Therefore, we create a function to compute the distance between each data point and each centroid.



In [ ]:
def compute_distance(data, centers, k):
    """
    """
    # Initialize distance
    distance = np.zeros((np.shape(data)[0], k))
    # Loop on n data points
    for i in range(0, np.shape(data)[0]):
        # Loop on k centroids
        for j in range(0, k):
            # Compute distance
            distance[i, j] = np.sqrt(np.sum(np.square(data[i, :] - centers[j, :])))
    return distance

We now need to create functions that assign each data point to a cluster. 

We also ned to define an **objective function** that we will seek to minimize.

Our objective is to minimize the sum of the square of the distance between each point and the closest centroid:

$obj = \sum_{j = 1}^k \sum_{i = 1}^{N_j} d(x^{(i)} , x^{(j)}) ^2$

where $x^{(i)}$ is the $i^{th}$ point in the cluster $j$, $x^{(j)}$ is the centroid of the cluster $j$, and $N_j$ is the number of points in the cluster $j$.

In [ ]:
def compute_clusters(distance):
    """
    """
    # Initialize clusters
    clusters = np.zeros(np.shape(distance)[0])
    # Loop on n data points
    for i in range(0, np.shape(distance)[0]):
        # Find closest centroid
        best = np.argmin(distance[i, :])
        # Assign data point to corresponding cluster
        clusters[i] = best
    return clusters

def compute_objective(distance, clusters):
    """
    """
    # Initialize objective
    objective = 0.0
    # Loop on n data points
    for i in range(0, np.shape(distance)[0]):
        # Add distance to the closest centroid
        objective = objective + distance[i, int(clusters[i])] ** 2.0
    return objective

After all points are assigned to a cluster, we will compute a new location of the centroid, which is just the mean of all the points affected to that cluster.

In [ ]:
def compute_centers(data, clusters, k):
    """
    """
    # Initialize centroids
    centers = np.zeros((k, np.shape(data)[1]))
    # Loop on clusters
    for i in range(0, k):
        # Select all data points in this cluster
        subdata = data[clusters == i, :]
        # If no data point in this cluster, generate randomly a new centroid
        if (np.shape(subdata)[0] == 0):
            centers[i, :] = init_centers(data, 1)
        else:
            # Compute the mean location of all data points in this cluster
            centers[i, :] = np.mean(subdata, axis=0)
    return centers

Let us put it all together.

In [ ]:
def our_kmeans(data, k):
    """
    """
    # Initialize centroids
    centers = init_centers(data, k)
    # Initialize objective function to square of the maximum distance between two data points times number of data points
    objective_old = np.shape(data)[0] * np.sum(np.square(np.max(data, axis=0) - np.min(data, axis=0)))
    # Initialize clusters
    clusters_old = np.zeros(np.shape(data)[0])
    # Start loop until convergence
    stop_alg = False
    while stop_alg == False:
        # Compute distance between data points and centroids
        distance = compute_distance(data, centers, k)
        # Get new clusters
        clusters_new = compute_clusters(distance)
        # get new value of objective function
        objective_new = compute_objective(distance, clusters_new)
        # If objective function stops decreasing, end loop
        if objective_new >= objective_old:
            return (clusters_old, objective_old, centers)
        else:
            # Update the locations of the centroids
            centers = compute_centers(data, clusters_new, k)
            objective_old = objective_new
            clusters_old = clusters_new

In [ ]:
# Let's run the code

k = 3
(clusters, objective, centers) = our_kmeans(data, k)
df["clusterID"] = clusters.astype('str')

# 3D scatter plot with matplotlib
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i, cluster_id in enumerate(sorted(data['clusterID'].unique())):
    mask = df['clusterID'] == cluster_id
    ax.scatter(df.loc[mask, 'norm.scatter'], 
               df.loc[mask, 'norm.red'], 
               df.loc[mask, 'norm.orange'],
               c=colors[i % len(colors)], label=f'Cluster {cluster_id}', s=5, alpha=0.6)

ax.set_xlabel('norm.scatter')
ax.set_ylabel('norm.red')
ax.set_zlabel('norm.orange')
ax.legend()
ax.set_box_aspect(None, zoom=0.85)
plt.show()